# Bistabilität von Standgewässern

![Alt text](https://passel2.unl.edu/image.php?uuid=0e40642a4afe&extension=png&display=ORIGINAL&v=1663679922)
*Photo by Stephen R. Carpenter.*

Wir schließen den Kreis vom Thema "Eutrophierung" zu Beginn der Lehrveranstaltung hin zum Thema "Stabilität dynamischer Systeme" am Ende der Lehrveranstaltung. Dazu versuchen wir, eines der Milestone-Paper von [Stephen R. Carpenter](https://en.wikipedia.org/wiki/Stephen_R._Carpenter), einem US-amerikanischen Gewässerökologen,  reproduzieren:

Carpenter, R. (2004): Eutrophication of aquatic ecosystems: Bistability and soil phosphorus, PNAS, 102(29), 10002–10005, https://doi.org/10.1073/pnas.0503959102.

Weil wir aber nicht viel Zeit und Muße haben, lassen wir ChatGPT den Job machen und versuchen, die Lösung nachzuvollziehen. Ich habe dazu folgenden Prompt genutzt:

> Please provide me the Python code required to solve the differential equations (with the parameters given in table 1) and required to reproduce the figures in the paper.

#### Einstiegsfrage

Es gibt in den Formeln für das DGL-System im Paper einen Fehler, den ChatGPT kommentarlos entdeckt und gefixt hat. Findest Du den Fehler?

#### Antwort

In Gleichung 4 muss es im Nenner P$^q$ heißen.

## Modellparameter

Zunächst übertragen wir die Modellparameter aus Tabelle 1 in unsere Umgebung.

#### Frage

Im Paper gibt es in der Tabelle zwei Fehler: einen in der Spalte `Symbol` und einen in der Spalte `Units`. Findest Du diese Fehler?

#### Antwort

Parameter `h` muss in Zeile 1 `b` heißen. Einheit von `s` ist y$^{-1}$.

#### Nun aber die Werte aus Tabelle 1

In [ ]:
b = 0.001      # sediment burial rate (yr^-1)
c = 0.00115    # runoff coefficient (yr^-1)
h = 0.15       # outflow rate (yr^-1)
s = 0.7        # sedimentation rate (yr^-1)
r = 0.019      # max recycling rate (yr^-1)
m = 2.4        # half-saturation phosphorus density (g m^-2)
q = 8

F_intensive = 31.6
H_intensive = 18.6

W_pre = 0.147
W_post = 1.55

## Gleichgewichtszustände im System

Bevor die dynamischen Szenarien nachgerechnet werden, wollen wir zunächst die Gleichgewichtszustände des Systems ermitteln. 

#### Aufgabe

Ermittle die Gleichgewichtszustände $U^*$, $P^*$ und $M^*$ mit Hilfe der Dir bekannten Vorgehensweise. Tipp: Löse erst für $U^*$ und eliminiere dann $M^*$. Es bleibt eine algebraische Gleichung für $P^*$, die wir leider nicht analytisch lösen können. Aber darum kümmmern wir uns später...Achtung: Wir identifizieren nur die Gleichgewichtszustände, führen aber  zunächst keine Stabilitätsanalyse durch.

#### Lösung

mit $L = W + F - H$ und $f(P) = \frac{P^q}{m^q+P^q}$ gilt

$$U^* = \frac{c}{L}$$

$$ G(U^*, P^*) = L - (s+h)P^* + \frac{r \cdot s \cdot P^* \cdot f(P^*)}{b+r \cdot f(P^*)} = 0$$

$$ M^* = \frac{s \cdot P^*}{b+r \cdot f(P^*)}$$

#### Noch eine Aufgabe

Die nicht-lineare Magie wird in dem DGL-System durch die Funktion `f(P)` bewirkt. 

#### Wie können wir die Nullstellen für $P^*$ finden?

Die Bestimmungsgleichung für $P^*$ lässt sich nur numerisch lösen (z.B. mit Newton-Verfahren). Wir nutzen dafür die Funktion `root_scalar` im Paket Modul `scipy.optimize`. Um Abbildung 2 reproduzieren zu können, wollen wir den Wert der Nullstellen in Abhängigkeit von der Kontrollvariable $L$ darstellen, also dem P-Input ins System.

In [ ]:
import numpy as np
from scipy.optimize import root_scalar

def f(P):
    return P**q / (m**q + P**q)

# Wie oben entwickelt
def equilibrium_equation(P, L):
    return (
        L
        - (s + h) * P
        + (r * s * P * f(P)) / (b + r * f(P))
    )


def compute_equilibria(L, Pmax=10):

    # "Nullstellen"
    roots = []

    # Der Bereich von P, den wir auf Nullstellen überprüfen wollen
    grid = np.logspace(-6, np.log10(Pmax), 2000)

    # Funktionswerte
    vals = equilibrium_equation(grid, L)

    for i in range(len(grid)-1):
        # Haben aufeinanderfolgende Werte von vals unterschiedliche Vorzeichen?
        # Dann muss eine Nullstelle dazwischen liegen!
        if vals[i] * vals[i+1] < 0:

            sol = root_scalar(
                equilibrium_equation,
                args=(L,),
                bracket=[grid[i], grid[i+1]]
            )

            roots.append(sol.root)

    return np.array(roots)

In [ ]:
L_values = np.linspace(0.0, 1.5, 300)

P_eq = []
M_eq = []

for L in L_values:

    roots = compute_equilibria(L)

    for P in roots:

        fp = f(P)
        M = s*P/(b + r*fp)

        P_eq.append([L, P])
        M_eq.append([L, M])

P_eq = np.array(P_eq)
M_eq = np.array(M_eq)

#### Erste Reproduktion von Abbildung 2

In [ ]:
import matplotlib.pyplot as plt

# Umstellung der Arrays, um eine durchgängige Linie zu zeichnen
sorter = np.argsort(P_eq[:, 1])
P_eq_plot = P_eq[sorter]
M_eq_plot = M_eq[sorter]

fig, ax = plt.subplots(2,1, figsize=(4,5))

plt.sca(ax[0])
plt.plot(P_eq_plot[:,0], P_eq_plot[:,1], "k-")
plt.ylabel("Water Phosphorus, g m⁻²")
plt.xlim(0,1.5)
plt.ylim(0,7)

plt.sca(ax[1])
plt.plot(M_eq_plot[:,0], M_eq_plot[:,1], "k-")
plt.xlabel("Phosphorus Input to Soil, g m⁻² y⁻¹")
plt.ylabel("Sediment Phosphorus, g m⁻²")
plt.xlim(0,1.5)
plt.ylim(0,900)

plt.tight_layout()

Diese Abbildung kommt der Abbildung 2 im Paper schon sehr nahe. Allerdings fällt auf, dass die konkreten Werte schlicht nicht hinhauen. Mein Verdacht: Irgendein Parameterwert in Tabelle 1 entspricht nicht dem Wert, der wirklich für Abbildung 2 im Paper genutzt wurde. Aber gut...mir sagte mal jemand: 

> "In jedem Paper ist mindestens ein Fehler in den Berechnungen enthalten. Meistens aber mehr."

Das Ergebnis ist dennoch bemerkenswert: Die Zahl der möglichen Gleichgewichtszustände hängt von $L$ ab. Bei sehr geringem Input gibt es nur ein (stabiles) Gleichgewicht, bei sehr hohem Input ebenfalls, aber dazwischen gibt es drei Gleichgewichtszustände! 

Allerdings fehlt die Stabilitätsanalyse, die für einen Gleichgewichtszustand jeweils entscheidet, ob es sich um ein stabiles oder instabiles Gleichgewicht handelt. In Abbildung 2 im Paper wird das instabile Gleichgewicht mit gestrichelter Linie gezeichnet. Auch ohne explizite Stabilitätsanalyse können wir auf Grundlage unserer bisherigen Erfahrung (z.B. Logistisches Wachstum mit Allee-Effekt) erwarten, dass das "mittlere" Gleichgewicht instabil und damit für die Betrachtung unerheblich ist. Der Code in der folgenden Zelle überprüft schlicht, wo die beiden Wendepunkte in der Kurve für $P$ liegen. In der Abbildung wird der Teil zwischen den Wendepunkten gestrichelt (nicht durchgezogen). Die strenge Mathematik der Stabilitätsanalyse ersparen wir und hingegen (es geht um die Eigenwerte der Jacobi-Matrix des DGL-Systems...).

In [ ]:
foundone=False
x = []
L = P_eq_plot[1:,0]
for i in range(len(L)-1):
    if not foundone:
        if L[i+1]<=L[i]:
            x.append(i)
            foundone=True
    else:
        if L[i+1]>=L[i]:
            x.append(i)
            break
x

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(4,5))

plt.sca(ax[0])
plt.plot(P_eq_plot[:(x[0]+1),0], P_eq_plot[:(x[0]+1),1], "k-")
plt.plot(P_eq_plot[x[0]:(x[1]+1),0], P_eq_plot[x[0]:(x[1]+1),1], "k--")
plt.plot(P_eq_plot[x[1]:,0], P_eq_plot[x[1]:,1], "k-")
plt.ylabel("Water Phosphorus, g m⁻²")
plt.xlim(0,1.5)
plt.ylim(0,7)

plt.sca(ax[1])
plt.plot(M_eq_plot[:(x[0]+1),0], M_eq_plot[:(x[0]+1),1], "k-")
plt.plot(M_eq_plot[x[0]:(x[1]+1),0], M_eq_plot[x[0]:(x[1]+1),1], "k--")
plt.plot(M_eq_plot[x[1]:,0], M_eq_plot[x[1]:,1], "k-")
plt.xlabel("Phosphorus Input to Soil, g m⁻² y⁻¹")
plt.ylabel("Sediment Phosphorus, g m⁻²")
plt.xlim(0,1.5)
plt.ylim(0,900)

plt.tight_layout()

#### Hinweis

Zur Nachbereitung könnt Ihr Euch auch nochmal diese Seite anschauen: https://passel2.unl.edu/view/lesson/d6c3e24cbc7e/4

## Dynamische Simulation des Systemverhaltens

Wir haben gezeigt, dass sich der Gleichgewichtszustand in Abhängkeit der Variable $L$ (P-Input) schlagartig ändern kann. Das sagt uns aber nichts darüber, wie lange es dauert, bis der neue Gleichgewichtszustand erreicht wird. Dazu benötigen wir eine dynamische Simulation des Systems, wir müssen also das DGL-System numerisch über die Zeit lösen.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp

# # -------------------------
# # Parameters
# # -------------------------
# b = 0.001
# c = 0.00115
# h = 0.15
# s = 0.7
# r = 0.019
# m = 2.4
# q = 8

# F_intensive = 31.6
# H_intensive = 18.6

# W_pre = 0.147
# W_post = 1.55




def rhs(t, y, forcing):
    U, P, M = y

    W, F, H = forcing(t)

    rec = r * M * f(P)

    dU = W + F - H - c * U
    dP = c * U - (s + h) * P + rec
    dM = s * P - b * M - rec

    return [dU, dP, dM]

In [ ]:
def forcing_sim1(t):

    if t < 100:
        return W_pre, 0.0, 0.0

    elif t < 200:
        return W_post, 0.0, 0.0

    elif t < 250:
        return W_post, F_intensive, H_intensive

    else:
        return W_post, H_intensive, H_intensive

In [ ]:
def forcing_sim2(t):

    if t < 100:
        return W_pre, 0.0, 0.0

    elif t < 200:
        return W_post, 0.0, 0.0

    elif t < 250:
        return W_post, F_intensive, H_intensive

    else:
        return W_pre, H_intensive, H_intensive

In [ ]:
y0 = [0.0, 0.01, 0.01]

t_eval = np.linspace(0, 5000, 20000)

sol1 = solve_ivp(
    lambda t,y: rhs(t,y,forcing_sim1),
    [0,5000],
    y0,
    t_eval=t_eval
)

sol2 = solve_ivp(
    lambda t,y: rhs(t,y,forcing_sim2),
    [0,5000],
    y0,
    t_eval=t_eval
)

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(10,8))

for sol, axis, title in [
    (sol1, ax[0], "Simulation 1"),
    (sol2, ax[1], "Simulation 2")
]:

    axis.semilogy(sol.t, sol.y[0], label="U (soil)")
    axis.semilogy(sol.t, sol.y[1], label="P (water)")
    axis.semilogy(sol.t, sol.y[2], label="M (sediment)")

    axis.set_title(title)
    axis.legend()

plt.tight_layout()
plt.show()